In [ ]:
pip install qiskit

In [ ]:
import qiskit
qiskit.__version__


In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, partial_trace, entropy
import matplotlib.pyplot as plt
import random

# ----------------------
# Parameters
# ----------------------
L = 6                           # number of qubits
L2 = L**2                      # heating steps
cooling_steps = 20           # maximum cooling iterations
beta = 50.0                     # inverse temperature
gate_set = ['1']  # 1 = ['cx', 'x', 'h']     # set I: {CNOT, NOT, H}
seed = 42
np.random.seed(seed)
random.seed(seed)

# ----------------------
# Initial Product State
# ----------------------
qc = QuantumCircuit(L)
# |000...0> is default initial state

# ----------------------
# Apply Heating: Random gates from I
# ----------------------
# def apply_random_gate(circuit, gate_set):
#     gate = random.choice(gate_set)
#     q1 = random.randint(0, L - 1)
    
#     if gate == 'cx':
#         q2 = random.randint(0, L - 1)
#         while q2 == q1:
#             q2 = random.randint(0, L - 1)
#         circuit.cx(q1, q2)
#     elif gate == 'x':
#         circuit.x(q1)
#     elif gate == 'h':
#         circuit.h(q1)

def apply_random_gate(qc, gate_set):
    gate = random.choice(gate_set)
    q1 = random.randint(0, qc.num_qubits - 1)
    q2 = random.randint(0, qc.num_qubits - 1)
    while q2 == q1:
        q2 = random.randint(0, qc.num_qubits - 1)
    qc.cx(q1, q2)
    if gate =='1' :
        qc.x(q1)
        qc.h(q1)

    elif gate == '2':
        qc.h(q1)
        qc.t(q1)


for _ in range(L2):
    apply_random_gate(qc, gate_set)

# Final entangled state after heating
state = Statevector.from_instruction(qc)

# ----------------------
# Entropy function (half-chain entanglement)
# ----------------------
def half_chain_entropy(statevec, L):
    rho_A = partial_trace(statevec, range(L//2, L))
    return entropy(rho_A)

# ----------------------
# Entanglement Cooling
# ----------------------
def cooling_algorithm(state, L, gate_set, beta, max_steps=1000):
    current_state = state
    current_entropy = half_chain_entropy(current_state, L)
    entropy_evolution = [current_entropy]
    accepted = 0

    

    for step in range(max_steps):
        # Propose a new gate and apply to clone of circuit
        qc_new = QuantumCircuit(L)
        #qc_new.statevec(current_state) set_statevector(current_state)
        
        gate = random.choice(gate_set)
        q1 = random.randint(0, L - 1)
        apply_random_gate(qc_new,gate)
        # if gate == 'cx':
        #     q2 = random.randint(0, L - 1)
        #     while q2 == q1:
        #         q2 = random.randint(0, L - 1)
        #     qc_new.cx(q1, q2)
        # elif gate == 'x':
        #     qc_new.x(q1)
        # elif gate == 'h':
        #     qc_new.h(q1)

        new_state = current_state.evolve(qc_new)
        new_entropy = half_chain_entropy(new_state, L)

        delta_S = new_entropy - current_entropy

        if delta_S <= 0 or np.random.rand() < np.exp(-beta * delta_S):
            current_state = new_state
            current_entropy = new_entropy
            accepted += 1
        
        entropy_evolution.append(current_entropy)
        
        if np.isclose(current_entropy, 0, atol=1e-4):
            print('required tolerance achieved')
            break
    
    return entropy_evolution, step + 1, accepted,current_state

# ----------------------
# Run Cooling
# ----------------------
entropy_trace, steps, acc,cool_st = cooling_algorithm(state, L, gate_set, beta, cooling_steps)

# ----------------------
# Plotting
# ----------------------
plt.figure(figsize=(8,5))
plt.plot(entropy_trace, label=f"β={beta}, steps={steps}")
plt.xlabel("Cooling Iteration")
plt.ylabel("Half-chain Entanglement Entropy")
plt.title(f"Entanglement Cooling (L={L}) with Gate Set: {gate_set}")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def entanglement_heating(L, gate_set, steps=None):
    if steps is None:
        steps = L**2  # default heating time

    qc = QuantumCircuit(L)
    for _ in range(steps):
        apply_random_gate(qc, gate_set)

    # Return the entangled state
    return Statevector.from_instruction(qc)


In [ ]:
# ----------------------
# Parameters
# ----------------------
L = 12                        # number of qubits
L2 = L**3                      # heating steps
cooling_steps = 5000          # maximum cooling iterations
beta = 1.0                     # inverse temperature
#gate_set = ['cx', 'x', 'h']     # set I: {CNOT, NOT, H}
gate_set = ['1']  # T-gate version (non-Clifford)

#seed = 42
np.random.seed()
random.seed()

# ----------------------
# Initial Product State
# ----------------------
qc2 = QuantumCircuit(L)
#state = entanglement_heating(L, gate_set)

for _ in range(L2):
    apply_random_gate(qc2, gate_set)

# Final entangled state after heating
state = Statevector.from_instruction(qc2)
print("Entropy after heating:", half_chain_entropy(state, L))

entropy_trace, steps, acc,f_state = cooling_algorithm(state, L, gate_set, beta, cooling_steps)
print("Entropy after cooling:", half_chain_entropy(f_state, L))
# ----------------------
# Plotting
# ----------------------
plt.figure(figsize=(8,5))
plt.plot(entropy_trace, label=f"β={beta}, steps={steps}")
plt.xlabel("Cooling Iteration")
plt.ylabel("Half-chain Entanglement Entropy")
plt.title(f"Entanglement Cooling (L={L}) with Gate Set: {gate_set}")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------
# Parameters
# ----------------------
L = 12                        # number of qubits
L2 = L**3                      # heating steps
cooling_steps = 5000          # maximum cooling iterations
beta = 1.0                     # inverse temperature
#gate_set = ['cx', 'x', 'h']     # set I: {CNOT, NOT, H}
gate_set = ['2']  # T-gate version (non-Clifford)

#seed = 42
np.random.seed()
random.seed()

# ----------------------
# Initial Product State
# ----------------------
qc2 = QuantumCircuit(L)
#state = entanglement_heating(L, gate_set)

for _ in range(L2):
    apply_random_gate(qc2, gate_set)

# Final entangled state after heating
state = Statevector.from_instruction(qc2)
print("Entropy after heating:", half_chain_entropy(state, L))

entropy_trace, steps, acc,f_state = cooling_algorithm(state, L, gate_set, beta, cooling_steps)
print("Entropy after cooling:", half_chain_entropy(f_state, L))
# ----------------------
# Plotting
# ----------------------
plt.figure(figsize=(8,5))
plt.plot(entropy_trace, label=f"β={beta}, steps={steps}")
plt.xlabel("Cooling Iteration")
plt.ylabel("Half-chain Entanglement Entropy")
plt.title(f"Entanglement Cooling (L={L}) with Gate Set: {gate_set}")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
L = 6
gate_set = ['cx', 'x', 'h']  # or ['cx', 'h', 't']
heated_state = entanglement_heating(L, gate_set)

print("Entropy after heating:", half_chain_entropy(heated_state, L))


In [ ]:
half_chain_entropy(state,L)

In [ ]:
L = 6
gate_set = ['cx', 'x', 'h']  # or ['cx', 'h', 't']
heated_state = entanglement_heating(L, gate_set)

print("Entropy after heating:", half_chain_entropy(heated_state, L))
# Build a random gate acting on 1 or 2 qubits
qc_gate = QuantumCircuit(L)
q1 = random.randint(0, L - 1)
gate = random.choice(gate_set)

if gate == 'cx':
    q2 = random.randint(0, L - 1)
    q1 = 
    while q2 == q1:
        q2 = random.randint(0, L - 1)
    qc_gate.cx(q1, q2)
elif gate == 'x':
    qc_gate.x(q1)
elif gate == 'h':
    qc_gate.h(q1)
elif gate == 't':
    qc_gate.t(q1)

# Apply it to the current state
new_state = heated_state.evolve(qc_gate)
print("Entropy after cooling:", half_chain_entropy(new_state, L))
